# Querying Data

Import Libraries and setting parameters

In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath('C:/Users/dagic/OneDrive/Documents/KAIM/Week_2/fintech-review-analytics'))

print('Path set. Python will now look in:', os.path.abspath('C:/Users/dagic/OneDrive/Documents/KAIM/Week_2/fintech-review-analytics'))

Path set. Python will now look in: C:\Users\dagic\OneDrive\Documents\KAIM\Week_2\fintech-review-analytics


In [2]:
import psycopg2
import pandas as pd

from src.db_helper import run_query



## Connection Creation

In [3]:
conn = psycopg2.connect(
    host="localhost",
    database="bank_reviews",
    user="admin_user",
    password="654123"
)

## View data

In [4]:
sql = "SELECT * FROM banks limit 10;"

run_query(sql, conn)

C:\Users\dagic\OneDrive\Documents\KAIM\Week_2\fintech-review-analytics\src\db_helper.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,bank_id,bank_name,app_name,bank_code
0,4,Commercial Bank of Ethiopia,CBE Mobile Banking,CBE
1,5,Bank of Abyssinia,BoA Mobile,Bank of Abyssinia
2,6,Dashen Bank,Dashen Bank Super App,Dashen Bank


In [5]:
sql = "SELECT * FROM reviews limit 10;"

run_query(sql, conn)



C:\Users\dagic\OneDrive\Documents\KAIM\Week_2\fintech-review-analytics\src\db_helper.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,review_id,bank_id,review_text,rating,review_date,sentiment_label,sentiment_score,identified_theme,source
0,1,6,best app so far. thank you,5,2026-05-15,positive,0.999848,Other,Google Play
1,2,6,Very Annoying App i tried to open virtual bank...,1,2026-05-14,negative,-0.999444,Account,Google Play
2,3,6,good,5,2026-05-14,positive,0.999816,Other,Google Play
3,4,6,good,5,2026-05-14,positive,0.999816,Other,Google Play
4,5,6,good app but it was doesnt work other bank tra...,5,2026-05-14,positive,-0.996920,UX,Google Play
5,6,6,good,5,2026-05-14,positive,0.999816,Other,Google Play
6,7,6,"i swear to god , By using this app, I won a Sa...",5,2026-05-14,positive,0.999208,Other,Google Play
7,8,6,good and easier to used,5,2026-05-14,positive,0.999848,Other,Google Play
8,9,6,bad mobile banking at all,1,2026-05-13,negative,-0.999807,Other,Google Play
9,10,6,very nice app.,5,2026-05-13,positive,0.999862,Other,Google Play


## Average rating per restaurant

In [6]:
# Average rating per banks

sql = """
SELECT r.bank_name, ROUND(AVG(v.rating), 2) AS avg_rating
FROM reviews v
JOIN banks r ON r.bank_id = v.bank_id
GROUP BY r.bank_name;
"""

run_query(sql, conn)


C:\Users\dagic\OneDrive\Documents\KAIM\Week_2\fintech-review-analytics\src\db_helper.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,bank_name,avg_rating
0,Commercial Bank of Ethiopia,4.08
1,Dashen Bank,3.92
2,Bank of Abyssinia,3.55


## Banks with the highest customer frustration levels

In [7]:
sql = """
    SELECT 
    b.bank_name,
    COUNT(*) AS total_reviews,
    COUNT(*) FILTER (WHERE r.rating <= 2) AS frustrated_reviews,
    ROUND(
        COUNT(*) FILTER (WHERE r.rating <= 2) * 100.0 / COUNT(*), 
        2
    ) AS frustration_rate
FROM reviews r
JOIN banks b ON r.bank_id = b.bank_id
GROUP BY b.bank_name
ORDER BY frustration_rate DESC;
"""

run_query(sql, conn)

C:\Users\dagic\OneDrive\Documents\KAIM\Week_2\fintech-review-analytics\src\db_helper.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,bank_name,total_reviews,frustrated_reviews,frustration_rate
0,Bank of Abyssinia,467,157,33.62
1,Dashen Bank,466,110,23.61
2,Commercial Bank of Ethiopia,457,82,17.94


## Detect of recurring login and authentication problems

In [8]:
query_login = """ 
SELECT 
    b.bank_id, b.bank_name,
    COUNT(*) AS issue_count
FROM reviews as r
JOIN banks as b
ON r.bank_id = b.bank_id
WHERE 
    LOWER(r.review_text) LIKE '%login%' OR
    LOWER(r.review_text) LIKE '%password%' OR
    LOWER(r.review_text) LIKE '%authentication%' OR
    LOWER(r.review_text) LIKE '%otp%' OR
    LOWER(r.review_text) LIKE '%verification%'
GROUP BY b.bank_id, b.bank_name
ORDER BY issue_count DESC;
"""

run_query(query_login, conn)

C:\Users\dagic\OneDrive\Documents\KAIM\Week_2\fintech-review-analytics\src\db_helper.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,bank_id,bank_name,issue_count
0,5,Bank of Abyssinia,15
1,6,Dashen Bank,8
2,4,Commercial Bank of Ethiopia,1


## Login & Authentication Problem

In [9]:
login_query = """
SELECT 
    bank_id,
    COUNT(*) AS issue_count
FROM reviews
WHERE 
    LOWER(review_text) LIKE '%login%' OR
    LOWER(review_text) LIKE '%password%' OR
    LOWER(review_text) LIKE '%authentication%' OR
    LOWER(review_text) LIKE '%otp%' OR
    LOWER(review_text) LIKE '%verification%'
GROUP BY bank_id
ORDER BY issue_count DESC;
"""

run_query(login_query, conn)

C:\Users\dagic\OneDrive\Documents\KAIM\Week_2\fintech-review-analytics\src\db_helper.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,bank_id,issue_count
0,5,15
1,6,8
2,4,1


## Customer Response to App Updates

In [10]:
update_query = """ 
SELECT 
    b.bank_name,
    review_date,
    AVG(sentiment_score) AS avg_sentiment
FROM reviews r
JOIN banks b ON r.bank_id = b.bank_id
GROUP BY b.bank_name, review_date
ORDER BY review_date;
"""

run_query(update_query, conn)

C:\Users\dagic\OneDrive\Documents\KAIM\Week_2\fintech-review-analytics\src\db_helper.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,bank_name,review_date,avg_sentiment
0,Bank of Abyssinia,2025-02-23,0.999840
1,Bank of Abyssinia,2025-02-26,-0.997277
2,Bank of Abyssinia,2025-03-01,-0.333256
3,Bank of Abyssinia,2025-03-02,-0.995671
4,Bank of Abyssinia,2025-03-03,0.004105
...,...,...,...
537,Bank of Abyssinia,2026-05-15,0.999847
538,Commercial Bank of Ethiopia,2026-05-15,-0.199324
539,Dashen Bank,2026-05-15,0.999848
540,Bank of Abyssinia,2026-05-16,-0.999812


## Frequently Requested Usability Improvements

In [11]:
usability_query = """ 
SELECT 
    identified_theme,
    COUNT(*) AS frequency
FROM reviews
GROUP BY identified_theme
ORDER BY frequency DESC;
"""

run_query(usability_query, conn)

C:\Users\dagic\OneDrive\Documents\KAIM\Week_2\fintech-review-analytics\src\db_helper.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,identified_theme,frequency
0,Other,1150
1,Stability,74
2,UX,72
3,Account,51
4,Features,43


## Frequently Requested Usability Improvements per bank

In [18]:
perbank_query = """ 
SELECT 
    b.bank_name,
    r.identified_theme,
    COUNT(*) AS frequency
FROM reviews r
JOIN banks b 
    ON r.bank_id = b.bank_id
GROUP BY 
    b.bank_name,
    r.identified_theme
ORDER BY frequency DESC;
"""
run_query(perbank_query, conn)


C:\Users\dagic\OneDrive\Documents\KAIM\Week_2\fintech-review-analytics\src\db_helper.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,bank_name,identified_theme,frequency
0,Commercial Bank of Ethiopia,Other,394
1,Bank of Abyssinia,Other,390
2,Dashen Bank,Other,366
3,Dashen Bank,UX,40
4,Bank of Abyssinia,Stability,38
5,Dashen Bank,Account,26
6,Commercial Bank of Ethiopia,UX,22
7,Bank of Abyssinia,Account,21
8,Commercial Bank of Ethiopia,Features,21
9,Dashen Bank,Stability,20


## Customer Satisfaction Comparison Across Banks

In [13]:
satisfaction_query = """ 
SELECT 
    b.bank_name,
    ROUND(AVG(r.rating), 2) AS avg_rating,
    ROUND(AVG(r.sentiment_score), 3) AS avg_sentiment,
    COUNT(*) AS total_reviews
FROM reviews r
JOIN banks b ON r.bank_id = b.bank_id
GROUP BY b.bank_name
ORDER BY avg_rating DESC;
"""

run_query(satisfaction_query, conn)

C:\Users\dagic\OneDrive\Documents\KAIM\Week_2\fintech-review-analytics\src\db_helper.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,bank_name,avg_rating,avg_sentiment,total_reviews
0,Commercial Bank of Ethiopia,4.08,0.396,457
1,Dashen Bank,3.92,0.338,466
2,Bank of Abyssinia,3.55,0.120,467


## Most Negative Reviews

In [14]:
negative_query = """ 
SELECT 
    b.bank_name,
    r.review_text,
    r.rating,
    r.sentiment_score
FROM reviews r
JOIN banks b ON r.bank_id = b.bank_id
WHERE r.rating <= 2
ORDER BY r.sentiment_score ASC;
"""

run_query(negative_query, conn)

C:\Users\dagic\OneDrive\Documents\KAIM\Week_2\fintech-review-analytics\src\db_helper.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,bank_name,review_text,rating,sentiment_score
0,Dashen Bank,The worst app. It needs updating everyday. Ouch 🤕,1,-0.999821
1,Dashen Bank,It is so boring.....,1,-0.999821
2,Commercial Bank of Ethiopia,it is not well functional. It always sluggish ...,1,-0.999820
3,Dashen Bank,Disgusting! You can't Buy any goods thinking y...,1,-0.999816
4,Dashen Bank,I have been trying repeatedly to send money to...,1,-0.999816
...,...,...,...,...
344,Commercial Bank of Ethiopia,Good application,2,0.999855
345,Commercial Bank of Ethiopia,nice,1,0.999855
346,Commercial Bank of Ethiopia,nice,1,0.999855
347,Dashen Bank,nice,1,0.999855


## Trend of Customer Sentiment Over Time

In [15]:
sentiment_query = """ 
SELECT 
    review_date,
    AVG(sentiment_score) AS avg_sentiment
FROM reviews
GROUP BY review_date
ORDER BY review_date;
"""

run_query(sentiment_query, conn)

C:\Users\dagic\OneDrive\Documents\KAIM\Week_2\fintech-review-analytics\src\db_helper.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,review_date,avg_sentiment
0,2025-02-23,0.999840
1,2025-02-26,-0.997277
2,2025-03-01,-0.333256
3,2025-03-02,-0.995671
4,2025-03-03,0.004105
...,...,...
348,2026-05-12,0.555472
349,2026-05-13,0.366572
350,2026-05-14,0.579032
351,2026-05-15,0.333641


## Most Frequent Complaint Themes per Bank

In [16]:
theme_query = """ 
SELECT 
    b.bank_name,
    r.identified_theme,
    COUNT(*) AS count
FROM reviews r
JOIN banks b ON r.bank_id = b.bank_id
GROUP BY b.bank_name, r.identified_theme
ORDER BY count DESC;
"""

run_query(theme_query, conn)

C:\Users\dagic\OneDrive\Documents\KAIM\Week_2\fintech-review-analytics\src\db_helper.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,bank_name,identified_theme,count
0,Commercial Bank of Ethiopia,Other,394
1,Bank of Abyssinia,Other,390
2,Dashen Bank,Other,366
3,Dashen Bank,UX,40
4,Bank of Abyssinia,Stability,38
5,Dashen Bank,Account,26
6,Commercial Bank of Ethiopia,UX,22
7,Bank of Abyssinia,Account,21
8,Commercial Bank of Ethiopia,Features,21
9,Dashen Bank,Stability,20


# Key Insights

## Commercial Bank of Ethiopia (CBE)

#### Finding:
•	Highest average rating: 4.08
•	Lower frustration rate
•	Close to none login and authentication issue
•	High request on UX improvement
•	Higher request on stability improvement
•	Require higher improvement on features
•	minimum improvement on account


#### Interpretation:
CBE demonstrates strong overall customer satisfaction, specially in reliability and authentication. However, users expect improvements in UX, and stability.


#### Recommendation:
•	Improve update communication 
•	Fix usability issues in transactions
•	Improve overall UX 


## Bank of Abyssinia (BOA)

#### Finding:
•	Lowest average rating: 3.55
•	Highest frustration rate in among others
•	High login and authentication issue
•	little request on UX improvement
•	Higher request on stability improvement
•	Require little improvement on features
•	high improvement on account


#### Interpretation:
BOA shows failures in core functionalities specially in authentication, account and stability. The reviews shows that users are dissatisfied with those gaps.


#### Recommendation:
•	fix login issues and authentication flow 
•	improve system stability and reduce errors
•	prioritize core functionality of the system than UX

## Dashen bank

#### Finding:
•	Mid-level rating: 3.92
•	Moderate frustration rate
•	Little login and authentication issue
•	Higher request on UX improvement
•	Low request on stability improvement
•	Require higher improvement on features
•	higher improvement on account


#### Interpretation:
Dashen bank performs moderately well than BOA with stability. Users review focus on UX improvement and features.


#### Recommendation:
•	Improve app navigation UX 
•	Focus on feature expansion

